# PatchCore + WideResNet50-2 Multilayer Training

This notebook runs the current best anomaly detector from the report on a larger labeled local split.

Model setup:
- PatchCore with a frozen pretrained `WideResNet50-2`
- multilayer feature bank from `layer2 + layer3`
- validation-threshold evaluation at the `0.95` normal quantile
- small comparison sweep around the report winner `topk_mb50k_r010`

In [ ]:
import os
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
BUNDLE_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "helpers" / "patchcore_wrn50_local.py").exists():
        BUNDLE_ROOT = candidate
        break

if BUNDLE_ROOT is None:
    raise RuntimeError("Could not locate the local WRN PatchCore experiment root.")

HELPERS_ROOT = BUNDLE_ROOT / "helpers"
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))

from patchcore_wrn50_local import (
    DEFAULT_SPLIT_CONFIG,
    DEFAULT_VARIANTS,
    WaferArrayDataset,
    attach_scores_to_metadata,
    auto_find_raw_pickle,
    defect_type_summary,
    load_wafer_array,
    prepare_dataset,
    resolve_data_root,
    resolve_device,
    resolve_output_root,
    run_patchcore_variant,
    set_seed,
    split_summary_wide,
)

In [ ]:
RAW_PICKLE = os.environ.get("WM811K_RAW_PICKLE")
IMAGE_SIZE = 64
SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0
DEVICE = "auto"
TEACHER_LAYERS = ["layer2", "layer3"]
PRETRAINED = True
FREEZE_BACKBONE = True
NORMALIZE_IMAGENET = True
BACKBONE_INPUT_SIZE = 224
QUERY_CHUNK_SIZE = 1024
MEMORY_CHUNK_SIZE = 4096
THRESHOLD_QUANTILE = 0.95
THRESHOLD_STRATEGY = "validation_f1"
MAX_VALIDATION_FALSE_POSITIVE_RATE = None
SPLIT_CONFIG = DEFAULT_SPLIT_CONFIG.copy()
SWEEP_VARIANTS = DEFAULT_VARIANTS.copy()
DATA_ROOT = resolve_data_root(BUNDLE_ROOT)
OUTPUT_ROOT = resolve_output_root(BUNDLE_ROOT)
ARTIFACT_DIR = OUTPUT_ROOT / "patchcore_wrn50_multilayer_120k_5pct"
PLOTS_DIR = ARTIFACT_DIR / "plots"
RUN_VARIANT_SWEEP = False
VARIANT_NAME_OVERRIDE = None
MIN_RECALL = 0.70
MAX_FALSE_POSITIVE_RATE = 0.03
MAX_AUTO_NORMAL_ANOMALY_RATE = 0.01
MIN_AUTO_ANOMALY_PRECISION = 0.60

set_seed(SEED)
device = resolve_device(DEVICE)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

metadata = None
metadata_path = None
raw_pickle = None
arrays_dir = None
DATASET_READY = False

try:
    raw_pickle = auto_find_raw_pickle(RAW_PICKLE)
    metadata_path = prepare_dataset(raw_pickle, DATA_ROOT, IMAGE_SIZE, SPLIT_CONFIG, seed=SEED, overwrite=False)
    metadata = pd.read_csv(metadata_path)
    _, arrays_dir = metadata_paths(DATA_ROOT, IMAGE_SIZE, SPLIT_CONFIG)
    DATASET_READY = True
except FileNotFoundError as exc:
    print(f"[WARNING] {exc}")
    print("[WARNING] Dataset previews will be skipped until the local WM811K source pickle is available.")

if DATASET_READY:
    display(split_summary_wide(metadata))
    display(defect_type_summary(metadata).head(18))

print("Bundle root:", BUNDLE_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Artifact dir:", ARTIFACT_DIR)
print("Raw pickle:", raw_pickle)
print("Metadata path:", metadata_path)
print("Using device:", device)
print("Run variant sweep:", RUN_VARIANT_SWEEP)
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

selected_variant_name = None
selected_result = None
bundle_summary = {}
sweep_results_df = pd.DataFrame()
val_scores_df = None
test_scores_df = None
threshold_sweep_df = None
pred_df = None
threshold = None
best_sweep = None
EVALUATION_READY = False

bundle_summary_path = ARTIFACT_DIR / "bundle_summary.json"
sweep_results_path = ARTIFACT_DIR / "patchcore_sweep_results.csv"

if RUN_VARIANT_SWEEP:
    if not DATASET_READY:
        print("[WARNING] Dataset inputs are unavailable, so the WRN50 PatchCore sweep cannot be rebuilt in this notebook run.")
    else:
        train_dataset = WaferArrayDataset(metadata_path, split="train", data_root=DATA_ROOT)
        val_dataset = WaferArrayDataset(metadata_path, split="val", data_root=DATA_ROOT)
        test_dataset = WaferArrayDataset(metadata_path, split="test", data_root=DATA_ROOT)

        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        variant_results = {}
        rows = []
        for variant in SWEEP_VARIANTS:
            print(f"\n=== Running variant: {variant['name']} ===")
            result = run_patchcore_variant(
                variant,
                train_dataset=train_dataset,
                val_loader=val_loader,
                test_loader=test_loader,
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                device=device,
                output_dir=ARTIFACT_DIR,
                seed=SEED,
                teacher_layers=TEACHER_LAYERS,
                pretrained=PRETRAINED,
                freeze_backbone=FREEZE_BACKBONE,
                backbone_input_size=BACKBONE_INPUT_SIZE,
                normalize_imagenet=NORMALIZE_IMAGENET,
                threshold_quantile=THRESHOLD_QUANTILE,
                threshold_strategy=THRESHOLD_STRATEGY,
                max_validation_false_positive_rate=MAX_VALIDATION_FALSE_POSITIVE_RATE,
                query_chunk_size=QUERY_CHUNK_SIZE,
                memory_chunk_size=MEMORY_CHUNK_SIZE,
            )
            variant_results[variant["name"]] = result
            rows.append(result["row"])

        sweep_results_df = pd.DataFrame(rows).sort_values(["f1", "auroc", "auprc"], ascending=False).reset_index(drop=True)
        sweep_results_df.to_csv(sweep_results_path, index=False)
        selected_variant_name = str(VARIANT_NAME_OVERRIDE or sweep_results_df.iloc[0]["name"])
        selected_result = variant_results[selected_variant_name]
        threshold = float(selected_result["threshold"])
        val_scores_df = selected_result["val_scores_df"]
        test_scores_df = selected_result["test_scores_df"]
        threshold_sweep_df = selected_result["threshold_sweep_df"]
        best_sweep = selected_result["best_sweep"]

        bundle_summary = {
            "selected_variant": selected_variant_name,
            "split_config": SPLIT_CONFIG,
            "raw_pickle": str(raw_pickle),
            "metadata_path": str(metadata_path),
            "threshold_strategy": THRESHOLD_STRATEGY,
            "variants": [variant["name"] for variant in SWEEP_VARIANTS],
        }
        bundle_summary_path.write_text(json.dumps(bundle_summary, indent=2), encoding="utf-8")

        pred_df = attach_scores_to_metadata(
            metadata[metadata["split"] == "test"].reset_index(drop=True),
            selected_result["test_scores_df"],
            threshold,
        )
        pred_df.to_csv(ARTIFACT_DIR / f"{selected_variant_name}_test_predictions.csv", index=False)
        EVALUATION_READY = True
else:
    if not bundle_summary_path.exists() or not sweep_results_path.exists():
        print("[WARNING] No saved WRN50 PatchCore artifact bundle was found.")
        print("[WARNING] Keep RUN_VARIANT_SWEEP=False for review-only mode, or set it to True to rebuild the local artifacts.")
    else:
        bundle_summary = json.loads(bundle_summary_path.read_text(encoding="utf-8"))
        sweep_results_df = pd.read_csv(sweep_results_path).sort_values("f1", ascending=False).reset_index(drop=True)
        selected_variant_name = str(VARIANT_NAME_OVERRIDE or bundle_summary["selected_variant"])
        selected_row = sweep_results_df.loc[sweep_results_df["name"] == selected_variant_name].iloc[0]
        variant_dir = ARTIFACT_DIR / selected_variant_name

        try:
            summary, val_scores_df, test_scores_df = load_variant_artifacts(ARTIFACT_DIR, selected_variant_name)
        except FileNotFoundError as exc:
            print(f"[WARNING] {exc}")
            print("[WARNING] Saved evaluation review will be skipped until the selected variant files exist.")
        else:
            threshold = float(summary["threshold"])
            threshold_sweep_path = variant_dir / "threshold_sweep.csv"
            threshold_sweep_df = (
                pd.read_csv(threshold_sweep_path)
                if threshold_sweep_path.exists()
                else build_threshold_sweep(test_scores_df)
            )
            best_sweep = summary.get("best_sweep") or {
                "threshold": float(selected_row.get("best_sweep_threshold", threshold)),
                "precision": float(selected_row.get("best_sweep_precision", summary.get("precision", 0.0))),
                "recall": float(selected_row.get("best_sweep_recall", summary.get("recall", 0.0))),
                "f1": float(selected_row.get("best_sweep_f1", summary.get("f1", 0.0))),
            }
            selected_result = {
                "metrics": summary.get(
                    "metrics_at_validation_threshold",
                    {
                        "precision": float(summary.get("precision", 0.0)),
                        "recall": float(summary.get("recall", 0.0)),
                        "f1": float(summary.get("f1", 0.0)),
                        "auroc": float(summary.get("auroc", 0.0)),
                        "auprc": float(summary.get("auprc", 0.0)),
                        "confusion_matrix": summary.get("confusion_matrix", [[0, 0], [0, 0]]),
                    },
                ),
                "best_sweep": best_sweep,
                "threshold": threshold,
                "val_scores_df": val_scores_df,
                "test_scores_df": test_scores_df,
            }

            predictions_path = ARTIFACT_DIR / f"{selected_variant_name}_test_predictions.csv"
            if predictions_path.exists():
                pred_df = pd.read_csv(predictions_path)
            elif DATASET_READY:
                pred_df = attach_scores_to_metadata(
                    metadata[metadata["split"] == "test"].reset_index(drop=True),
                    test_scores_df,
                    threshold,
                )
                pred_df.to_csv(predictions_path, index=False)
            else:
                pred_df = None

            EVALUATION_READY = True

if EVALUATION_READY:
    print(f"Selected variant: {selected_variant_name}")
    display(sweep_results_df)
else:
    print("[WARNING] No WRN50 evaluation tables are available to display in this notebook run.")


In [ ]:
if EVALUATION_READY and selected_result is not None:
    if EVALUATION_READY and selected_result is not None:
        if EVALUATION_READY and selected_result is not None:
            metrics = selected_result["metrics"]
            best_sweep = selected_result["best_sweep"]
            threshold = float(selected_result["threshold"])

            metrics_df = pd.DataFrame(
                [
                    {"metric": "precision", "value": metrics["precision"]},
                    {"metric": "recall", "value": metrics["recall"]},
                    {"metric": "f1", "value": metrics["f1"]},
                    {"metric": "auroc", "value": metrics["auroc"]},
                    {"metric": "auprc", "value": metrics["auprc"]},
                    {"metric": "threshold", "value": threshold},
                ]
            )

            display(metrics_df)
            display(
                pd.DataFrame(
                    metrics["confusion_matrix"],
                    index=["true_normal", "true_anomaly"],
                    columns=["pred_normal", "pred_anomaly"],
                )
            )
            print(
                f"Best sweep threshold: {best_sweep['threshold']:.6f} | "
                f"precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}"
            )
        else:
            print('WRN50 selected-variant metrics are unavailable because the local artifact bundle is missing.')
    else:
        print('WRN50 selected-variant metrics are unavailable because the local artifact bundle is missing.')
else:
    print('WRN50 selected-variant metrics are unavailable because the local artifact bundle is missing.')


In [ ]:
if EVALUATION_READY and selected_result is not None:
    if EVALUATION_READY and selected_result is not None:
        if EVALUATION_READY and selected_result is not None:
            plot_df = sweep_results_df.copy()
            plot_df["label"] = plot_df["name"] + "\n" + plot_df["reduction"] + ", mb=" + plot_df["memory_bank_size"].astype(str)

            fig, axes = plt.subplots(1, 2, figsize=(15, 5))
            axes[0].barh(plot_df["label"], plot_df["f1"], color="#277da1")
            axes[0].set_title("WRN PatchCore Sweep: Validation-Threshold F1")
            axes[0].set_xlabel("F1")
            axes[0].invert_yaxis()

            axes[1].barh(plot_df["label"], plot_df["auroc"], color="#90be6d")
            axes[1].set_title("WRN PatchCore Sweep: AUROC")
            axes[1].set_xlabel("AUROC")
            axes[1].invert_yaxis()

            plt.tight_layout()
            plt.show()

            val_scores_df = selected_result["val_scores_df"]
            test_scores_df = selected_result["test_scores_df"]
            threshold_sweep_df = selected_result["threshold_sweep_df"]

            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].hist(val_scores_df["score"], bins=30, alpha=0.8, color="#4d908e")
            axes[0].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
            axes[0].set_title(f"Validation Score Distribution\n{selected_variant_name}")
            axes[0].legend()

            axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal")
            axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly")
            axes[1].axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
            axes[1].set_title(f"Test Score Distribution\n{selected_variant_name}")
            axes[1].legend()

            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(8, 4))
            plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
            plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
            plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
            plt.axvline(threshold, color="red", linestyle="--", label=f"validation threshold = {threshold:.4f}")
            plt.axvline(best_sweep["threshold"], color="green", linestyle=":", label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
            plt.title(f"Threshold Sweep on Test Split\n{selected_variant_name}")
            plt.xlabel("Anomaly-score threshold")
            plt.ylabel("Metric value")
            plt.legend()
            plt.show()
        else:
            print('WRN50 sweep plots are unavailable because the local artifact bundle is missing.')
    else:
        print('WRN50 sweep plots are unavailable because the local artifact bundle is missing.')
else:
    print('WRN50 sweep plots are unavailable because the local artifact bundle is missing.')


## Failure Analysis

This section attaches the selected PatchCore scores to the held-out test metadata and highlights the strongest false positives and false negatives at the validation-derived threshold.

In [ ]:
if EVALUATION_READY and selected_result is not None:
    if EVALUATION_READY and selected_result is not None:
        if EVALUATION_READY and selected_result is not None:
            test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)
            scored_test_df = attach_scores_to_metadata(test_metadata, selected_result["test_scores_df"], threshold)
            scored_test_df.to_csv(ARTIFACT_DIR / f"{selected_variant_name}_test_predictions.csv", index=False)

            error_summary = (
                scored_test_df.groupby(["error_type", "defect_type"])
                .size()
                .rename("count")
                .reset_index()
                .sort_values(["error_type", "count"], ascending=[True, False])
            )
            display(error_summary.head(20))

            def plot_examples(frame, title, ascending, top_n=5):
                sample = frame.sort_values("score", ascending=ascending).head(top_n)
                if sample.empty:
                    print(f"No rows available for {title}.")
                    return
                fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.2))
                if len(sample) == 1:
                    axes = [axes]
                for ax, (_, row) in zip(axes, sample.iterrows()):
                    wafer = load_wafer_array(DATA_ROOT, row["array_path"])
                    ax.imshow(wafer, cmap="viridis")
                    ax.set_title(f"{row['defect_type']}\nscore={row['score']:.3f}")
                    ax.axis("off")
                fig.suptitle(title)
                plt.tight_layout()
                plt.show()

            false_positive_df = scored_test_df[scored_test_df["error_type"] == "false_positive"]
            false_negative_df = scored_test_df[scored_test_df["error_type"] == "false_negative"]
            plot_examples(false_positive_df, "Top False Positives", ascending=False)
            plot_examples(false_negative_df, "Top False Negatives", ascending=True)
        else:
            print('WRN50 failure-analysis outputs are unavailable because the local artifact bundle is missing.')
    else:
        print('WRN50 failure-analysis outputs are unavailable because the local artifact bundle is missing.')
else:
    print('WRN50 failure-analysis outputs are unavailable because the local artifact bundle is missing.')


## Dataset Sanity View

The old dataset-helper notebook is folded into the main experiment flow here so the split summary and a few representative wafer maps live next to the training and evaluation cells.


In [ ]:
if not DATASET_READY:
    print("[WARNING] Dataset sanity plots are unavailable because the raw dataset inputs could not be prepared locally.")
else:
    display(split_summary(metadata))
    display(split_summary_wide(metadata))

    sample_specs = [
        ("train", 0, "Train normal"),
        ("train", 1, "Train anomaly"),
        ("val", 0, "Val normal"),
        ("val", 1, "Val anomaly"),
        ("test", 0, "Test normal"),
        ("test", 1, "Test anomaly"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, (split_name, is_anomaly, title) in zip(axes.ravel(), sample_specs):
        rows = metadata[(metadata["split"] == split_name) & (metadata["is_anomaly"] == is_anomaly)]
        if rows.empty:
            ax.axis("off")
            continue
        row = rows.iloc[0]
        wafer = load_wafer_array(DATA_ROOT, row["array_path"])
        ax.imshow(wafer, cmap="viridis")
        ax.set_title(f"{title}\n{row['defect_type']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Threshold Policies

The saved threshold-policy review now lives directly in this notebook so we can compare alternate operating points without jumping to a second file.


In [ ]:
if not EVALUATION_READY or selected_result is None or val_scores_df is None or test_scores_df is None:
    print("[WARNING] Threshold-policy analysis is unavailable because the selected WRN50 score bundle is missing.")
else:
    single_threshold_df = build_single_threshold_policy_table(
        val_scores_df,
        test_scores_df,
        current_threshold=float(threshold),
        min_recall=MIN_RECALL,
        max_false_positive_rate=MAX_FALSE_POSITIVE_RATE,
    )
    display(single_threshold_df.sort_values("test_f1", ascending=False).reset_index(drop=True).round(4))

    val_sweep_df = build_threshold_sweep(val_scores_df)
    validation_f1_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1", "threshold"].iloc[0])
    fp_cap_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"fp_cap_{MAX_FALSE_POSITIVE_RATE:.2%}", "threshold"].iloc[0])
    recall_floor_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}", "threshold"].iloc[0])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(val_sweep_df["threshold"], val_sweep_df["precision"], label="Validation precision", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["recall"], label="Validation recall", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["f1"], label="Validation F1", linewidth=2)
    ax.axvline(float(threshold), color="black", linestyle="--", linewidth=1.5, label="Current threshold")
    ax.axvline(validation_f1_threshold, color="tab:green", linestyle=":", linewidth=1.5, label="Validation F1 threshold")
    ax.axvline(fp_cap_threshold, color="tab:red", linestyle=":", linewidth=1.5, label="FP-cap threshold")
    ax.axvline(recall_floor_threshold, color="tab:blue", linestyle=":", linewidth=1.5, label="Recall-floor threshold")
    ax.set_title(f"Validation Threshold Sweep: {selected_variant_name}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric value")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    review_policy_df = build_review_policy_summary(
        val_scores_df,
        test_scores_df,
        max_auto_normal_anomaly_rate=MAX_AUTO_NORMAL_ANOMALY_RATE,
        min_auto_anomaly_precision=MIN_AUTO_ANOMALY_PRECISION,
    )
    display(review_policy_df.round(4))

    best_f1_row = single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1"].iloc[0]
    recall_row = single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}"].iloc[0]
    review_row = review_policy_df.iloc[0]
    message = f"""
### Working Guidance

- **Reduce false negatives:** use the recall-floor threshold near `{recall_row['threshold']:.6f}`. On the saved test split it reaches recall `{recall_row['test_recall']:.3f}`, but false positives rise to `{int(recall_row['test_fp'])}`.
- **Reduce false defect calls:** use the FP-capped or validation-F1 threshold. The validation-F1 threshold `{best_f1_row['threshold']:.6f}` cuts test false positives to `{int(best_f1_row['test_fp'])}` and lifts test F1 to `{best_f1_row['test_f1']:.3f}`.
- **Reduce both kinds of automatic mistakes:** use the review band. With the current defaults, scores below `{review_row['low_threshold']:.6f}` are auto-normal, scores at or above `{review_row['high_threshold']:.6f}` are auto-anomaly, and the middle goes to review.
- With that review band on the saved test split, auto-normal wafers have anomaly rate `{review_row['test_auto_normal_anomaly_rate']:.3%}`, auto-anomaly wafers have precision `{review_row['test_auto_anomaly_precision']:.3%}`, and `{review_row['test_review_rate']:.2%}` of wafers go to review.
"""
    display(Markdown(message))


## Dataset Sanity View

The old dataset-helper notebook is folded into the main experiment flow here so the split summary and a few representative wafer maps live next to the training and evaluation cells.


In [ ]:
if not DATASET_READY:
    print("[WARNING] Dataset sanity plots are unavailable because the raw dataset inputs could not be prepared locally.")
else:
    display(split_summary(metadata))
    display(split_summary_wide(metadata))

    sample_specs = [
        ("train", 0, "Train normal"),
        ("train", 1, "Train anomaly"),
        ("val", 0, "Val normal"),
        ("val", 1, "Val anomaly"),
        ("test", 0, "Test normal"),
        ("test", 1, "Test anomaly"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, (split_name, is_anomaly, title) in zip(axes.ravel(), sample_specs):
        rows = metadata[(metadata["split"] == split_name) & (metadata["is_anomaly"] == is_anomaly)]
        if rows.empty:
            ax.axis("off")
            continue
        row = rows.iloc[0]
        wafer = load_wafer_array(DATA_ROOT, row["array_path"])
        ax.imshow(wafer, cmap="viridis")
        ax.set_title(f"{title}\n{row['defect_type']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Threshold Policies

The saved threshold-policy review now lives directly in this notebook so we can compare alternate operating points without jumping to a second file.


In [ ]:
if not EVALUATION_READY or selected_result is None or val_scores_df is None or test_scores_df is None:
    print("[WARNING] Threshold-policy analysis is unavailable because the selected WRN50 score bundle is missing.")
else:
    single_threshold_df = build_single_threshold_policy_table(
        val_scores_df,
        test_scores_df,
        current_threshold=float(threshold),
        min_recall=MIN_RECALL,
        max_false_positive_rate=MAX_FALSE_POSITIVE_RATE,
    )
    display(single_threshold_df.sort_values("test_f1", ascending=False).reset_index(drop=True).round(4))

    val_sweep_df = build_threshold_sweep(val_scores_df)
    validation_f1_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1", "threshold"].iloc[0])
    fp_cap_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"fp_cap_{MAX_FALSE_POSITIVE_RATE:.2%}", "threshold"].iloc[0])
    recall_floor_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}", "threshold"].iloc[0])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(val_sweep_df["threshold"], val_sweep_df["precision"], label="Validation precision", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["recall"], label="Validation recall", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["f1"], label="Validation F1", linewidth=2)
    ax.axvline(float(threshold), color="black", linestyle="--", linewidth=1.5, label="Current threshold")
    ax.axvline(validation_f1_threshold, color="tab:green", linestyle=":", linewidth=1.5, label="Validation F1 threshold")
    ax.axvline(fp_cap_threshold, color="tab:red", linestyle=":", linewidth=1.5, label="FP-cap threshold")
    ax.axvline(recall_floor_threshold, color="tab:blue", linestyle=":", linewidth=1.5, label="Recall-floor threshold")
    ax.set_title(f"Validation Threshold Sweep: {selected_variant_name}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric value")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    review_policy_df = build_review_policy_summary(
        val_scores_df,
        test_scores_df,
        max_auto_normal_anomaly_rate=MAX_AUTO_NORMAL_ANOMALY_RATE,
        min_auto_anomaly_precision=MIN_AUTO_ANOMALY_PRECISION,
    )
    display(review_policy_df.round(4))

    best_f1_row = single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1"].iloc[0]
    recall_row = single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}"].iloc[0]
    review_row = review_policy_df.iloc[0]
    message = f"""
### Working Guidance

- **Reduce false negatives:** use the recall-floor threshold near `{recall_row['threshold']:.6f}`. On the saved test split it reaches recall `{recall_row['test_recall']:.3f}`, but false positives rise to `{int(recall_row['test_fp'])}`.
- **Reduce false defect calls:** use the FP-capped or validation-F1 threshold. The validation-F1 threshold `{best_f1_row['threshold']:.6f}` cuts test false positives to `{int(best_f1_row['test_fp'])}` and lifts test F1 to `{best_f1_row['test_f1']:.3f}`.
- **Reduce both kinds of automatic mistakes:** use the review band. With the current defaults, scores below `{review_row['low_threshold']:.6f}` are auto-normal, scores at or above `{review_row['high_threshold']:.6f}` are auto-anomaly, and the middle goes to review.
- With that review band on the saved test split, auto-normal wafers have anomaly rate `{review_row['test_auto_normal_anomaly_rate']:.3%}`, auto-anomaly wafers have precision `{review_row['test_auto_anomaly_precision']:.3%}`, and `{review_row['test_review_rate']:.2%}` of wafers go to review.
"""
    display(Markdown(message))


## Dataset Sanity View

The old dataset-helper notebook is folded into the main experiment flow here so the split summary and a few representative wafer maps live next to the training and evaluation cells.


In [ ]:
if not DATASET_READY:
    print("[WARNING] Dataset sanity plots are unavailable because the raw dataset inputs could not be prepared locally.")
else:
    display(split_summary(metadata))
    display(split_summary_wide(metadata))

    sample_specs = [
        ("train", 0, "Train normal"),
        ("train", 1, "Train anomaly"),
        ("val", 0, "Val normal"),
        ("val", 1, "Val anomaly"),
        ("test", 0, "Test normal"),
        ("test", 1, "Test anomaly"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, (split_name, is_anomaly, title) in zip(axes.ravel(), sample_specs):
        rows = metadata[(metadata["split"] == split_name) & (metadata["is_anomaly"] == is_anomaly)]
        if rows.empty:
            ax.axis("off")
            continue
        row = rows.iloc[0]
        wafer = load_wafer_array(DATA_ROOT, row["array_path"])
        ax.imshow(wafer, cmap="viridis")
        ax.set_title(f"{title}\n{row['defect_type']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Threshold Policies

The saved threshold-policy review now lives directly in this notebook so we can compare alternate operating points without jumping to a second file.


In [ ]:
if not EVALUATION_READY or selected_result is None or val_scores_df is None or test_scores_df is None:
    print("[WARNING] Threshold-policy analysis is unavailable because the selected WRN50 score bundle is missing.")
else:
    single_threshold_df = build_single_threshold_policy_table(
        val_scores_df,
        test_scores_df,
        current_threshold=float(threshold),
        min_recall=MIN_RECALL,
        max_false_positive_rate=MAX_FALSE_POSITIVE_RATE,
    )
    display(single_threshold_df.sort_values("test_f1", ascending=False).reset_index(drop=True).round(4))

    val_sweep_df = build_threshold_sweep(val_scores_df)
    validation_f1_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1", "threshold"].iloc[0])
    fp_cap_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"fp_cap_{MAX_FALSE_POSITIVE_RATE:.2%}", "threshold"].iloc[0])
    recall_floor_threshold = float(single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}", "threshold"].iloc[0])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(val_sweep_df["threshold"], val_sweep_df["precision"], label="Validation precision", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["recall"], label="Validation recall", linewidth=2)
    ax.plot(val_sweep_df["threshold"], val_sweep_df["f1"], label="Validation F1", linewidth=2)
    ax.axvline(float(threshold), color="black", linestyle="--", linewidth=1.5, label="Current threshold")
    ax.axvline(validation_f1_threshold, color="tab:green", linestyle=":", linewidth=1.5, label="Validation F1 threshold")
    ax.axvline(fp_cap_threshold, color="tab:red", linestyle=":", linewidth=1.5, label="FP-cap threshold")
    ax.axvline(recall_floor_threshold, color="tab:blue", linestyle=":", linewidth=1.5, label="Recall-floor threshold")
    ax.set_title(f"Validation Threshold Sweep: {selected_variant_name}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric value")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    review_policy_df = build_review_policy_summary(
        val_scores_df,
        test_scores_df,
        max_auto_normal_anomaly_rate=MAX_AUTO_NORMAL_ANOMALY_RATE,
        min_auto_anomaly_precision=MIN_AUTO_ANOMALY_PRECISION,
    )
    display(review_policy_df.round(4))

    best_f1_row = single_threshold_df.loc[single_threshold_df["policy"] == "validation_f1"].iloc[0]
    recall_row = single_threshold_df.loc[single_threshold_df["policy"] == f"recall_floor_{MIN_RECALL:.2f}"].iloc[0]
    review_row = review_policy_df.iloc[0]
    message = f"""
### Working Guidance

- **Reduce false negatives:** use the recall-floor threshold near `{recall_row['threshold']:.6f}`. On the saved test split it reaches recall `{recall_row['test_recall']:.3f}`, but false positives rise to `{int(recall_row['test_fp'])}`.
- **Reduce false defect calls:** use the FP-capped or validation-F1 threshold. The validation-F1 threshold `{best_f1_row['threshold']:.6f}` cuts test false positives to `{int(best_f1_row['test_fp'])}` and lifts test F1 to `{best_f1_row['test_f1']:.3f}`.
- **Reduce both kinds of automatic mistakes:** use the review band. With the current defaults, scores below `{review_row['low_threshold']:.6f}` are auto-normal, scores at or above `{review_row['high_threshold']:.6f}` are auto-anomaly, and the middle goes to review.
- With that review band on the saved test split, auto-normal wafers have anomaly rate `{review_row['test_auto_normal_anomaly_rate']:.3%}`, auto-anomaly wafers have precision `{review_row['test_auto_anomaly_precision']:.3%}`, and `{review_row['test_review_rate']:.2%}` of wafers go to review.
"""
    display(Markdown(message))
